# 🚀 Routing101 - Multi-Signal Video Retrieval on Kaggle

Notebook này được tối ưu hoá để chạy toàn bộ **Web App Tìm Kiếm Video Đa Tín Hiệu (FastAPI + Static UI)** và **Pipeline** trên Kaggle với 2 bộ dataset:
- **Dataset 1**: `https://www.kaggle.com/datasets/nguyenthanghuu/aic2026-dataset` (Chứa `Keyframes`, `Videos`, `map-keyframes`)
- **Dataset 2**: `https://www.kaggle.com/datasets/lmcu000/rrqbundle` (Chứa `siglib_embed`, `captions`, `caption_embed`, `ocr`, `transcripts`, `transcript_embed`, `summaries`, `summary_embed`, `filtered_object`)

### ⚙️ Cấu hình bắt buộc trên thanh công cụ Kaggle:
1. **Accelerator**: Chọn **`GPU T4 x2`** (Khuyên dùng - tương thích PyTorch 100% sm_75) hoặc CPU / P100.
2. **Internet**: Chọn `ON`
3. **Data > Input**: Đã thêm cả 2 dataset trên.

## 📦 BƯỚC 1: Tải Mã Nguồn Repo & Cài đặt Dependencies

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# 0. Luôn đưa cwd về /kaggle/working trước để tránh lỗi 'deleted working directory'
try:
    os.chdir("/kaggle/working")
except Exception:
    pass
%cd /kaggle/working

WORKSPACE_DIR = Path("/kaggle/working/Routing101")
REPO_URL = "https://github.com/hoangducbao/UnoptimalLonx.git"

# 1. Clone mã nguồn Routing101 về Kaggle nếu chưa có, hoặc git pull mới nhất
if not (WORKSPACE_DIR / "backend").exists():
    print(f"📥 Đang tải mã nguồn từ {REPO_URL}...")
    if WORKSPACE_DIR.exists():
        shutil.rmtree(WORKSPACE_DIR, ignore_errors=True)
    !git clone {REPO_URL} /kaggle/working/Routing101
else:
    print("🔄 Đang đồng bộ và cập nhật mã nguồn mới nhất từ repo...")
    !cd /kaggle/working/Routing101 && git remote set-url origin {REPO_URL} && git fetch origin main && git reset --hard origin/main

# 2. Chuyển vào thư mục repository và thiết lập sys.path
%cd /kaggle/working/Routing101
try:
    os.chdir("/kaggle/working/Routing101")
except Exception:
    pass

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# 3. Ghi đè trực tiếp backend/models.py tương thích GPU T4/P100/CPU an toàn
models_py_code = '''import numpy as np
import torch
from PIL import Image
from . import config

def _select_device() -> str:
    if torch.cuda.is_available():
        try:
            cap = torch.cuda.get_device_capability()
            if cap[0] >= 7:
                torch.zeros(1, device="cuda")
                return "cuda"
            else:
                print(f"[Device Warning] GPU compute capability {cap[0]}.{cap[1]} < 7.0 (e.g. Tesla P100). Falling back to CPU.")
                return "cpu"
        except Exception as e:
            print(f"[Device Warning] CUDA check failed ({e}). Falling back to CPU.")
            return "cpu"
    return "cpu"

DEVICE = _select_device()
_siglip2 = None

def load_siglip2():
    global _siglip2
    if _siglip2 is None:
        from transformers import AutoModel, AutoProcessor
        model = AutoModel.from_pretrained(config.SIGLIP2_MODEL_ID).to(DEVICE).eval()
        processor = AutoProcessor.from_pretrained(config.SIGLIP2_MODEL_ID)
        _siglip2 = (model, processor)
    return _siglip2

def encode_text_siglip2(texts: list) -> np.ndarray:
    model, processor = load_siglip2()
    inputs = processor(text=texts, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.get_text_features(**inputs)
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return feats.float().cpu().numpy().astype("float32")

def encode_image_siglip2(images: list) -> np.ndarray:
    model, processor = load_siglip2()
    inputs = processor(images=images, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.get_image_features(**inputs)
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return feats.float().cpu().numpy().astype("float32")

def is_image_query(query) -> bool:
    return isinstance(query, Image.Image)

def siglip2_query_vec(query) -> np.ndarray:
    if is_image_query(query):
        return encode_image_siglip2([query])[0]
    return encode_text_siglip2([query])[0]
'''
(WORKSPACE_DIR / "backend/models.py").write_text(models_py_code, encoding="utf-8")

# 4. Ghi đè trực tiếp backend/main.py hoàn chỉnh
main_py_code = '''from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles

from . import config
from .es_indexing import ensure_all_fuzzy_indices
from .models import DEVICE, load_siglip2
from .routes import export, facets, hierarchy, neighbors, playback, query_image, search, trake
from .search import asr as asr_mod
from .search import caption as cap_mod
from .search import keyframe as kf
from .search import summary as sum_mod

@asynccontextmanager
async def lifespan(app: FastAPI):
    config.tune_thread_pools(DEVICE)
    print(f"[startup] device={DEVICE} cpu_budget={config.CPU_BUDGET}")

    print("[startup] loading SigLIP2 text/image tower…")
    load_siglip2()

    print("[startup] Keyframe — SigLIP2 frame index")
    kf.build_frame_index(config.FRAME_SIGLIP2_GLOB)
    print("[startup] Keyframe — CLIP frame index")
    kf.build_frame_index(config.FRAME_CLIP_GLOB)

    print("[startup] ASR — SigLIP2 index")
    asr_mod.build_siglip_asr_index()
    print("[startup] Caption — SigLIP2 index")
    cap_mod.build_siglip_caption_index()
    print("[startup] Summary — embeddings + SigLIP2 index")
    sum_mod.build_siglip_summary_index()

    try:
        print("[startup] ASR/Caption/OCR/Summary — Elasticsearch")
        ensure_all_fuzzy_indices()
    except Exception as e:
        print(f"[startup] Elasticsearch warning: {e}")

    print("[startup] all signals ready")
    yield

app = FastAPI(title="Routing101 by MiLF", lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

app.include_router(search.router)
app.include_router(facets.router)
app.include_router(neighbors.router)
app.include_router(playback.router)
app.include_router(query_image.router)
app.include_router(trake.router)
app.include_router(hierarchy.router)
app.include_router(export.router)

config.THUMBNAIL_ROOT.mkdir(parents=True, exist_ok=True)
config.VIDEO_DIR.mkdir(parents=True, exist_ok=True)
app.mount("/media/keyframes", StaticFiles(directory=config.THUMBNAIL_ROOT), name="keyframes")
app.mount("/media/video", StaticFiles(directory=config.VIDEO_DIR), name="video")
app.mount("/app", StaticFiles(directory=config.REPO_ROOT / "frontend", html=True), name="frontend")

@app.get("/")
def root():
    from fastapi.responses import RedirectResponse
    return RedirectResponse("/app/")
'''
(WORKSPACE_DIR / "backend/main.py").write_text(main_py_code, encoding="utf-8")

# 5. Ghi đè trực tiếp backend/es_indexing.py an toàn tuyệt đối
es_indexing_code = '''import pandas as pd
from . import config
from .es_client import get_es_client

def ensure_asr_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_ASR): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_ASR, mappings={"properties": {"video_id": {"type": "keyword"}, "segment_id": {"type": "integer"}, "start_sec": {"type": "float"}, "text": {"type": "text"}}})
        def _docs():
            if not config.TRANSCRIPTS_DIR.exists(): return
            for csv_path in sorted(config.TRANSCRIPTS_DIR.glob("*.csv")):
                if csv_path.name == "manifest.csv": continue
                df = pd.read_csv(csv_path)
                if df.empty: continue
                video_id = csv_path.stem
                for _, r in df.iterrows():
                    text = r.get("text") or r.get("transcript") or ""
                    yield {"_index": config.ES_INDEX_ASR, "_id": f"{video_id}_{int(r.get('segment_id', 0))}", "_source": {"video_id": video_id, "segment_id": int(r.get("segment_id", 0)), "start_sec": float(r.get("start_sec", 0.0)), "text": str(text)}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES ASR Info] {e}")
        return False

def ensure_caption_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_CAPTION): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_CAPTION, mappings={"properties": {"video_id": {"type": "keyword"}, "frame_id": {"type": "integer"}, "text": {"type": "text"}}})
        def _docs():
            if not config.CAPTIONING_DIR.exists(): return
            for csv_path in sorted(config.CAPTIONING_DIR.glob("*.csv")):
                if csv_path.name == "manifest.csv": continue
                df = pd.read_csv(csv_path)
                if df.empty: continue
                video_id = csv_path.stem
                for _, r in df.iterrows():
                    v_id = r.get("video_id", video_id)
                    text = r.get("caption_text") or r.get("text") or r.get("caption") or ""
                    yield {"_index": config.ES_INDEX_CAPTION, "_id": f"{v_id}_{int(r['frame_id'])}", "_source": {"video_id": v_id, "frame_id": int(r["frame_id"]), "text": str(text)}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES Caption Info] {e}")
        return False

def ensure_ocr_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_OCR): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_OCR, mappings={"properties": {"video_id": {"type": "keyword"}, "frame_id": {"type": "integer"}, "text": {"type": "text"}}})
        def _docs():
            if not config.OCR_DIR.exists(): return
            for csv_path in sorted(config.OCR_DIR.glob("*.csv")):
                if csv_path.name.startswith("run_manifest"): continue
                video_id = csv_path.stem
                df = pd.read_csv(csv_path)
                if df.empty or "frame_id" not in df.columns: continue
                text_col = "text" if "text" in df.columns else df.columns[-1]
                grouped = df.groupby("frame_id")[text_col].apply(lambda s: " ".join(str(t) for t in s if pd.notna(t)))
                for frame_id, text in grouped.items():
                    if not text.strip(): continue
                    yield {"_index": config.ES_INDEX_OCR, "_id": f"{video_id}_{int(frame_id)}", "_source": {"video_id": video_id, "frame_id": int(frame_id), "text": text}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES OCR Info] {e}")
        return False

def ensure_summary_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_SUMMARY): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_SUMMARY, mappings={"properties": {"video_id": {"type": "keyword"}, "text": {"type": "text"}}})
        def _docs():
            if not config.SUMMARY_DIR.exists(): return
            for txt_path in sorted(config.SUMMARY_DIR.glob("*.txt")):
                video_id = txt_path.stem
                text = txt_path.read_text(encoding="utf-8").strip()
                if not text: continue
                yield {"_index": config.ES_INDEX_SUMMARY, "_id": video_id, "_source": {"video_id": video_id, "text": text}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES Summary Info] {e}")
        return False

def ensure_all_fuzzy_indices():
    for fn in [ensure_asr_fuzzy_index, ensure_caption_fuzzy_index, ensure_ocr_fuzzy_index, ensure_summary_fuzzy_index]:
        try:
            fn()
        except Exception as e:
            print(f"[ES Info] {fn.__name__}: {e}")
'''
(WORKSPACE_DIR / "backend/es_indexing.py").write_text(es_indexing_code, encoding="utf-8")

# 6. Ghi đè trực tiếp keyframe.py chuẩn an toàn cao
keyframe_code = '''import glob as glob_mod
import faiss
import numpy as np
import pandas as pd
from cachetools import TTLCache

from .. import config
from ..common import l2_normalize, query_hash, video_id_from_filename
from ..models import is_image_query, siglip2_query_vec

try:
    import clip_encoder
except Exception:
    clip_encoder = None

_FRAME_INDICES: dict = {}

def build_frame_index(glob_pattern: str):
    npy_paths = sorted(glob_mod.glob(glob_pattern))
    if not npy_paths:
        print(f"[Keyframe Warning] No .npy files matched: {glob_pattern}. Initializing empty index.")
        dim = 768 if "siglip" in glob_pattern.lower() else 512
        index = faiss.IndexFlatIP(dim)
        result = (index, pd.DataFrame(columns=["video_id", "frame_id"]))
        _FRAME_INDICES[glob_pattern] = result
        return result

    all_vecs = []
    lookup_rows = []
    for npy_path in npy_paths:
        video_id = video_id_from_filename(npy_path, ("_viclip768", "_clip32", "_siglip768", "_siglip2"))
        vecs = np.load(npy_path).astype("float32")
        if vecs.ndim == 1:
            vecs = vecs.reshape(1, -1)
        for row_idx in range(len(vecs)):
            lookup_rows.append({"video_id": video_id, "frame_id": row_idx})
        all_vecs.append(vecs)

    matrix = l2_normalize(np.vstack(all_vecs).astype("float32"))
    index = faiss.IndexFlatIP(matrix.shape[1])
    index.add(matrix)
    result = (index, pd.DataFrame(lookup_rows))
    _FRAME_INDICES[glob_pattern] = result
    return result

def _get_frame_index(glob_pattern: str):
    if glob_pattern not in _FRAME_INDICES:
        build_frame_index(glob_pattern)
    return _FRAME_INDICES[glob_pattern]

def _search_frame(index, lookup_df, qvec: np.ndarray, k: int) -> pd.DataFrame:
    if index.ntotal == 0 or lookup_df.empty:
        return pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "n"])
    q = l2_normalize(qvec.reshape(1, -1))
    n = min(k, index.ntotal)
    scores, ids = index.search(q, n)
    valid_mask = (ids[0] >= 0) & (ids[0] < len(lookup_df))
    valid_ids = ids[0][valid_mask]
    if len(valid_ids) == 0:
        return pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "n"])
    results = lookup_df.iloc[valid_ids].copy().reset_index(drop=True)
    results["score"] = scores[0][valid_mask]
    results["rank"] = np.arange(1, len(results) + 1)
    results["n"] = results["frame_id"] + 1
    return results[["rank", "score", "video_id", "frame_id", "n"]]

_siglip2_cache = TTLCache(maxsize=256, ttl=300)
_clip_cache = TTLCache(maxsize=256, ttl=300)

def search_siglip2_frame(query, k: int = config.FETCH_K) -> pd.DataFrame:
    cache_key = (query_hash(query), k)
    if cache_key in _siglip2_cache:
        return _siglip2_cache[cache_key]
    index, lookup_df = _get_frame_index(config.FRAME_SIGLIP2_GLOB)
    qvec = siglip2_query_vec(query)
    result = _search_frame(index, lookup_df, qvec, k)
    _siglip2_cache[cache_key] = result
    return result

_siglip2_cache = TTLCache(maxsize=256, ttl=300)

def search_siglip2_frame(query, k: int = config.FETCH_K) -> pd.DataFrame:
    cache_key = (query_hash(query), k)
    if cache_key in _siglip2_cache:
        return _siglip2_cache[cache_key]
    index, lookup_df = _get_frame_index(config.FRAME_SIGLIP2_GLOB)
    qvec = siglip2_query_vec(query)
    result = _search_frame(index, lookup_df, qvec, k)
    _siglip2_cache[cache_key] = result
    return result
'''
(WORKSPACE_DIR / "backend/search/keyframe.py").write_text(keyframe_code, encoding="utf-8")

# 7. Ghi đè trực tiếp backend/es_client.py dùng 127.0.0.1 và hỗ trợ force_new
es_client_code = '''from elasticsearch import Elasticsearch
from . import config

_client = None

def get_es_client(force_new: bool = False):
    global _client
    if _client is None or force_new:
        host = str(config.ES_HOST).replace("localhost", "127.0.0.1")
        if not host.startswith("http"):
            host = f"http://{host}"
        _client = Elasticsearch(
            hosts=[host],
            request_timeout=5,
            verify_certs=False,
            ssl_show_warn=False,
            meta_header=False,
        )
    return _client
'''
(WORKSPACE_DIR / "backend/es_client.py").write_text(es_client_code, encoding="utf-8")

# 8. Ghi đè trực tiếp backend/main.py loại bỏ hoàn toàn CLIP để tiết kiệm RAM
main_py_code = '''from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from . import config
from .es_indexing import ensure_all_fuzzy_indices
from .models import DEVICE, load_siglip2
from .routes import export, facets, hierarchy, neighbors, playback, query_image, search, trake
from .search import asr as asr_mod
from .search import caption as cap_mod
from .search import keyframe as kf
from .search import summary as sum_mod

@asynccontextmanager
async def lifespan(app: FastAPI):
    config.tune_thread_pools(DEVICE)
    print(f"[startup] device={DEVICE} cpu_budget={config.CPU_BUDGET}")
    print("[startup] loading SigLIP2 text/image tower…")
    load_siglip2()
    print("[startup] Keyframe — SigLIP2 frame index")
    kf.build_frame_index(config.FRAME_SIGLIP2_GLOB)
    print("[startup] ASR — SigLIP2 index")
    asr_mod.build_siglip_asr_index()
    print("[startup] Caption — SigLIP2 index")
    cap_mod.build_siglip_caption_index()
    print("[startup] Summary — embeddings + SigLIP2 index")
    sum_mod.build_siglip_summary_index()
    try:
        print("[startup] ASR/Caption/OCR/Summary — Elasticsearch")
        ensure_all_fuzzy_indices()
    except Exception as e:
        print(f"[startup] Elasticsearch warning: {e}")
    print("[startup] all signals ready")
    yield

class NoCacheStaticFiles(StaticFiles):
    def file_response(self, *args, **kwargs):
        response = super().file_response(*args, **kwargs)
        response.headers["Cache-Control"] = "no-cache"
        return response

app = FastAPI(title="Routing101 by MiLF", lifespan=lifespan)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
app.include_router(search.router)
app.include_router(facets.router)
app.include_router(neighbors.router)
app.include_router(playback.router)
app.include_router(query_image.router)
app.include_router(trake.router)
app.include_router(hierarchy.router)
app.include_router(export.router)

config.THUMBNAIL_ROOT.mkdir(parents=True, exist_ok=True)
config.VIDEO_DIR.mkdir(parents=True, exist_ok=True)
app.mount("/media/keyframes", StaticFiles(directory=config.THUMBNAIL_ROOT), name="keyframes")
app.mount("/media/video", StaticFiles(directory=config.VIDEO_DIR), name="video")
app.mount("/app", NoCacheStaticFiles(directory=config.REPO_ROOT / "frontend", html=True), name="frontend")

@app.get("/")
def root():
    from fastapi.responses import RedirectResponse
    return RedirectResponse("/app/")
'''
(WORKSPACE_DIR / "backend/main.py").write_text(main_py_code, encoding="utf-8")

# 9. Cài đặt các thư viện cần thiết (loại bỏ multilingual-clip và timm giúp tiết kiệm RAM và tăng tốc cài đặt)
!pip install -q --upgrade pip
!pip install -q fastapi uvicorn python-multipart cachetools "elasticsearch>=8.11,<8.13" faiss-cpu
!pip install -q transformers pillow tqdm pandas numpy
!pip install -q pycloudflared python-dotenv requests ultralytics

print("✅ Bước 1: Mã nguồn đã được vá hoàn chỉnh & Dependencies đã sẵn sàng!")

## 🐘 BƯỚC 2: Cài đặt và khởi chạy Elasticsearch nền (Không cần Docker)

In [ ]:
import os
import time
import requests
import subprocess
from pathlib import Path

print("=" * 60)
print("🐘 CÀI ĐẶT VÀ KHỞI ĐỘNG ELASTICSEARCH NỀN")
print("=" * 60)

# 1. Dọn dẹp sạch tiến trình cũ & file lock
!pkill -9 -f elasticsearch 2>/dev/null || true
!rm -f /opt/elasticsearch-8.11.0/data/node.lock 2>/dev/null || true

# 2. Tải và giải nén Elasticsearch 8.11.0 (nếu chưa tải)
if not Path("/opt/elasticsearch-8.11.0").exists():
    print("📥 Đang tải Elasticsearch 8.11.0 (khoảng 30s)...")
    !wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-8.11.0-linux-x86_64.tar.gz -O /tmp/es.tar.gz
    !tar -xzf /tmp/es.tar.gz -C /opt/
    !rm -f /tmp/es.tar.gz
    print("✅ Đã giải nén vào /opt/elasticsearch-8.11.0")

# 3. Cấu hình Development Mode an toàn (localhost 127.0.0.1 để bypass bootstrap check)
es_yml = '''cluster.name: kaggle-routing101
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
'''
Path("/opt/elasticsearch-8.11.0/config/elasticsearch.yml").write_text(es_yml, encoding="utf-8")

# Cấu hình Heap 512MB - 1GB
heap_opts_dir = Path("/opt/elasticsearch-8.11.0/config/jvm.options.d")
heap_opts_dir.mkdir(parents=True, exist_ok=True)
(heap_opts_dir / "heap.options").write_text("-Xms512m\n-Xmx1g\n", encoding="utf-8")

# 4. Phân quyền cho esuser và tạo file log trực tiếp trong /kaggle/working/
!useradd -m -s /bin/bash esuser 2>/dev/null || true
!chown -R esuser:esuser /opt/elasticsearch-8.11.0
!touch /kaggle/working/elasticsearch.log && chmod 666 /kaggle/working/elasticsearch.log
!chmod -R 777 /tmp

# 5. Khởi động Elasticsearch nền và lưu log trực tiếp vào /kaggle/working/elasticsearch.log
print("🚀 Đang khởi động tiến trình Elasticsearch...")
es_log_file = open("/kaggle/working/elasticsearch.log", "a")
subprocess.Popen(
    ["su", "-", "esuser", "-c", "/opt/elasticsearch-8.11.0/bin/elasticsearch"],
    stdout=es_log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 6. Chờ cổng 9200 sẵn sàng
print("⏳ Đang đợi Elasticsearch sẵn sàng...", end="", flush=True)
es_ready = False
for attempt in range(35):
    try:
        r = requests.get("http://127.0.0.1:9200", timeout=1)
        if r.status_code == 200:
            info = r.json()
            print(f"\n✅ THÀNH CÔNG! Elasticsearch (v{info.get('version', {}).get('number')}) đã sẵn sàng tại http://127.0.0.1:9200")
            print(f"📄 File log Elasticsearch: /kaggle/working/elasticsearch.log")
            es_ready = True
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(1)

if not es_ready:
    print("\n\n❌ Elasticsearch chưa sẵn sàng. Log chi tiết từ /kaggle/working/elasticsearch.log:")
    print("=" * 60)
    !cat /kaggle/working/elasticsearch.log | tail -n 35
    print("=" * 60)


## ⚡ BƯỚC 3: Tự động phát hiện & Ánh xạ đường dẫn siêu nhanh (Deep Pruned Auto-Discovery)

In [ ]:
import os
import csv
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

print("=" * 70)
print("📂 ĐANG QUÉT CẤU TRÚC /kaggle/input (FAST PRUNED DISCOVERY)...")
print("=" * 70)

TARGET_NAMES = {
    "keyframes", "videos", "video", "map-keyframes", "map_keyframes",
    "siglib_embed", "siglip_embed", "siglip2_embed",
    "captions", "caption", "caption_embed", "captions_embed",
    "ocr", "summaries", "summary", "summary_embed", "summaries_embed",
    "transcripts", "transcript", "transcript_embed", "transcripts_embed",
    "clip-features-32", "clip_features_32",
    "filtered_object", "filtered_objects", "objects", "object_detection"
}

all_found_dirs = {}

if KAGGLE_INPUT.exists():
    for root, dirs, _ in os.walk(KAGGLE_INPUT):
        root_path = Path(root)
        name_lower = root_path.name.lower()
        
        # Nếu thư mục hiện tại là mục tiêu, ghi nhận và dừng đào sâu
        if name_lower in TARGET_NAMES:
            all_found_dirs[name_lower] = root_path
            dirs.clear()  # Dừng không đào sâu vào các thư mục con
            continue
        
        # Bỏ qua các thư mục con bắt đầu bằng video id
        dirs[:] = [d for d in dirs if not d.startswith(("L0", "L1", "L2", "v_"))]
        
        rel = root_path.relative_to(KAGGLE_INPUT)
        if len(rel.parts) > 0 and len(rel.parts) <= 3:
            indent = "   " * (len(rel.parts) - 1)
            print(f"{indent}└── 📁 {root_path.name}")
else:
    print("⚠️ Không tìm thấy thư mục /kaggle/input/")

print("=" * 70)

# Hàm lấy thư mục tìm thấy theo thứ tự ưu tiên
def get_target(keys: list, fallback: Path) -> Path:
    for k in keys:
        if k.lower() in all_found_dirs:
            return all_found_dirs[k.lower()]
    return fallback

# 1. Từ nguyenthanghuu/aic2026-dataset (Keyframes, Videos, map-keyframes)
KEYFRAMES_DIR = get_target(["keyframes"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/Keyframes")
VIDEOS_DIR = get_target(["videos", "video"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/Videos")
MAP_KEYFRAMES_DIR = get_target(["map-keyframes", "map_keyframes"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/map-keyframes")

# Tạo thư mục dummy nếu dataset thiếu keyframes/videos raw để StaticFiles không bao giờ lỗi
if not KEYFRAMES_DIR.exists():
    KEYFRAMES_DIR = Path("/kaggle/working/keyframes_dummy")
    KEYFRAMES_DIR.mkdir(parents=True, exist_ok=True)
if not VIDEOS_DIR.exists():
    VIDEOS_DIR = Path("/kaggle/working/videos_dummy")
    VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Từ lmcu000/rrqbundle (Embeddings & CSVs)
SIGLIP_DIR = get_target(["siglib_embed", "siglip_embed", "siglip2_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/siglib_embed")
CAPTION_EMBED_DIR = get_target(["caption_embed", "captions_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/caption_embed")
CAPTIONS_DIR = get_target(["captions", "caption"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/captions")
OCR_DIR = get_target(["ocr"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/ocr")
SUMMARY_EMBED_DIR = get_target(["summary_embed", "summaries_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/summary_embed")
SUMMARIES_DIR = get_target(["summaries", "summary"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/summaries")
TRANSCRIPT_EMBED_DIR = get_target(["transcript_embed", "transcripts_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/transcript_embed")
TRANSCRIPTS_DIR = get_target(["transcripts", "transcript"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/transcripts")

# 3. Object Detection (OD) Filter
FILTERED_OBJECT_DIR = get_target(["filtered_object", "filtered_objects", "objects", "object_detection"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/filtered_object")
CLASS_VOCAB_CSV = FILTERED_OBJECT_DIR / "class_vocab.csv" if FILTERED_OBJECT_DIR.exists() else Path("/kaggle/working/class_vocab.csv")

# Nếu có per-video OD CSV nhưng chưa có class_vocab.csv, tự động tạo ngay
if FILTERED_OBJECT_DIR.exists() and not CLASS_VOCAB_CSV.exists():
    print("🔨 Đang tự động tạo class_vocab.csv từ filtered_object/*.csv...")
    names = set()
    for p in FILTERED_OBJECT_DIR.glob("*.csv"):
        if p.name == "class_vocab.csv": continue
        try:
            with open(p, "r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames and "class_name" in reader.fieldnames:
                    for row in reader:
                        raw = row.get("class_name")
                        if raw:
                            names.add(" ".join(str(raw).strip().lower().split()))
        except Exception:
            continue
    CLASS_VOCAB_CSV = Path("/kaggle/working/class_vocab.csv")
    with open(CLASS_VOCAB_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_name"])
        for name in sorted(names):
            writer.writerow([name])
    print(f"✅ Đã tạo {len(names)} unique OD classes tại {CLASS_VOCAB_CSV}")

# Đảm bảo thư mục lưu index và summary embed trên ổ ghi được
Path("/kaggle/working/index").mkdir(parents=True, exist_ok=True)
if not SUMMARY_EMBED_DIR.exists():
    SUMMARY_EMBED_DIR = Path("/kaggle/working/summary_embed")
    SUMMARY_EMBED_DIR.mkdir(parents=True, exist_ok=True)

# 4. Ghi đè file backend/config.py với nội dung hoàn chỉnh tự động nhận diện đường dẫn (SigLIP2-only)
config_py_content = f'''import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

FETCH_K = 100
DISPLAY_N = 30
RRF_K = 60
NEIGHBOR_WINDOW = 7
TOP_G_DEFAULT = 5

DATASET_MODE = "AIC"
SIGLIP2_MODEL_ID = "google/siglip2-base-patch16-384"

FRAME_SIGLIP2_GLOB = "{SIGLIP_DIR}/*.npy"

ASR_EMBED_DIR = Path("{TRANSCRIPT_EMBED_DIR}")
TRANSCRIPTS_DIR = Path("{TRANSCRIPTS_DIR}")

CAPTIONING_DIR = Path("{CAPTIONS_DIR}")
SIGLIP_CAPTION_DIR = Path("{CAPTION_EMBED_DIR}")

OCR_DIR = Path("{OCR_DIR}")

FILTERED_OBJECT_DIR = Path("{FILTERED_OBJECT_DIR}")
CLASS_VOCAB_CSV = Path("{CLASS_VOCAB_CSV}")

SUMMARY_DIR = Path("{SUMMARIES_DIR}")
SUMMARY_EMBED_DIR = Path("{SUMMARY_EMBED_DIR}")

MAP_KEYFRAMES_DIR = Path("{MAP_KEYFRAMES_DIR}")
THUMBNAIL_ROOT = Path("{KEYFRAMES_DIR}")
VIDEO_DIR = Path("{VIDEOS_DIR}")

INDEX_PREFIX = "routing101"

try:
    SUMMARY_EMBED_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    pass

REPO_ROOT = Path(__file__).resolve().parent.parent
INDEX_DIR = Path("/kaggle/working/index")
PIPELINE_DIR = REPO_ROOT / "pipeline"
ASR_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_asr"
CAPTION_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_caption"
SUMMARY_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_summary"
ASR_INDEX_DIR.mkdir(parents=True, exist_ok=True)
CAPTION_INDEX_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_INDEX_DIR.mkdir(parents=True, exist_ok=True)
SIGLIP_ASR_FAISS = ASR_INDEX_DIR / "siglip_asr_flat_ip.index"
SIGLIP_ASR_META = ASR_INDEX_DIR / "meta_siglip_asr.csv"
SIGLIP_CAPTION_FAISS = CAPTION_INDEX_DIR / "siglip_caption_flat_ip.index"
SIGLIP_CAPTION_META = CAPTION_INDEX_DIR / "meta_siglip_caption.csv"
SIGLIP_SUMMARY_FAISS = SUMMARY_INDEX_DIR / "siglip_summary_flat_ip.index"
SIGLIP_SUMMARY_META = SUMMARY_INDEX_DIR / "meta_siglip_summary.csv"

ES_HOST = "http://127.0.0.1:9200"
ES_INDEX_ASR = "asr_segments"
ES_INDEX_CAPTION = "caption_frames"
ES_INDEX_OCR = "ocr_frames"
ES_INDEX_SUMMARY = "summary_videos"

CPU_BUDGET = max(1, (os.cpu_count() or 4) - 2)

def tune_thread_pools(device: str) -> None:
    import faiss
    import torch
    if device == "cpu":
        torch.set_num_threads(CPU_BUDGET)
    torch.set_num_interop_threads(1)
    faiss.omp_set_num_threads(CPU_BUDGET)
'''

Path("/kaggle/working/Routing101/backend/config.py").write_text(config_py_content, encoding="utf-8")
print("✅ Đã ghi đè cấu hình hoàn hảo vào /kaggle/working/Routing101/backend/config.py!")

# 5. Hiển thị trạng thái
status = lambda p: "✅ SẴN SÀNG" if p.exists() else "⚠️ KHÔNG TÌM THẤY"
print("=" * 70)
print(f"• Keyframes Root         : {status(KEYFRAMES_DIR)} -> {KEYFRAMES_DIR}")
print(f"• Videos Root            : {status(VIDEOS_DIR)} -> {VIDEOS_DIR}")
print(f"• Map-Keyframes CSV      : {status(MAP_KEYFRAMES_DIR)} -> {MAP_KEYFRAMES_DIR}")
print(f"• SigLIP2 Embeddings     : {status(SIGLIP_DIR)} -> {SIGLIP_DIR}")
print(f"• Captions CSV           : {status(CAPTIONS_DIR)} -> {CAPTIONS_DIR}")
print(f"• Caption Embeddings     : {status(CAPTION_EMBED_DIR)} -> {CAPTION_EMBED_DIR}")
print(f"• OCR CSV                : {status(OCR_DIR)} -> {OCR_DIR}")
print(f"• Object Detection (OD)  : {status(FILTERED_OBJECT_DIR)} -> {FILTERED_OBJECT_DIR}")
print(f"• OD Class Vocabulary    : {status(CLASS_VOCAB_CSV)} -> {CLASS_VOCAB_CSV}")
print(f"• Transcripts CSV        : {status(TRANSCRIPTS_DIR)} -> {TRANSCRIPTS_DIR}")
print(f"• Transcript Embeddings  : {status(TRANSCRIPT_EMBED_DIR)} -> {TRANSCRIPT_EMBED_DIR}")
print(f"• Summaries TXT          : {status(SUMMARIES_DIR)} -> {SUMMARIES_DIR}")
print(f"• Summary Embeddings     : {status(SUMMARY_EMBED_DIR)} -> {SUMMARY_EMBED_DIR}")
print("=" * 70)
print("🚀 Cấu hình đường dẫn hoàn tất!")

## 🩺 BƯỚC 4: Kiểm tra nhanh Search Engine (Diagnostics)

In [ ]:
import sys
import time
import importlib
import requests
from pathlib import Path

WORKSPACE_DIR = Path("/kaggle/working/Routing101")
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
%cd /kaggle/working/Routing101

print("=" * 60)
print("🔍 KIỂM TRA HỆ THỐNG VÀ SEARCH ENGINE")
print("=" * 60)

# 1a. Kiểm tra Elasticsearch qua HTTP REST trực tiếp
try:
    resp = requests.get("http://127.0.0.1:9200", timeout=2)
    if resp.status_code == 200:
        info = resp.json()
        print(f"✅ [1/3] Elasticsearch (HTTP 9200): Kết nối OK (Cluster: {info.get('cluster_name')}, v{info.get('version', {}).get('number')})")
    else:
        print(f"⚠️ [1/3] Elasticsearch HTTP returned status {resp.status_code}")
except Exception as e:
    print(f"⚠️ [1/3] Elasticsearch (HTTP 9200) không phản hồi: {e}")
    print("   → Hãy chạy lại Bước 2 trước!")

# 1b. Kiểm tra Elasticsearch Python Client (force-reload module để tránh singleton cũ)
try:
    import backend.config
    importlib.reload(backend.config)
    import backend.es_client
    importlib.reload(backend.es_client)
    from backend.es_client import get_es_client
    
    es = get_es_client(force_new=True)
    host_used = backend.config.ES_HOST
    print(f"   [DEBUG] ES_HOST config = {host_used}")
    
    es_info = es.info()
    cluster = es_info.get('cluster_name', 'unknown')
    version = es_info.get('version', {}).get('number', 'unknown')
    print(f"✅ [2/3] Elasticsearch (Python Client): OK (Cluster: {cluster}, v{version})")
except Exception as e:
    print(f"⚠️ [2/3] Elasticsearch (Python Client): LỖI - {type(e).__name__}: {e}")
    print(f"   → Kiểm tra ES_HOST trong backend/config.py và đảm bảo dùng http://127.0.0.1:9200")

# 2. Kiểm tra tìm kiếm mẫu qua SigLIP2
try:
    from backend.search import keyframe as kf_mod
    t0 = time.time()
    sample_res = kf_mod.search_siglip2_frame("person riding a bicycle", k=5)
    dt = (time.time() - t0) * 1000
    print(f"✅ [3/3] SigLIP2 Vector Search: Thành công ({len(sample_res)} kết quả trong {dt:.1f}ms)")
except Exception as e:
    print(f"⚠️ [3/3] Search test warning: {e}")

print("=" * 60)
print("🎉 Kiểm tra hoàn tất, chuyển sang Bước 5 để mở Web App!")

## 🌐 BƯỚC 5: Khởi chạy Web App & Mở Public URL qua Cloudflare Tunnel

In [ ]:
import subprocess
import time
import re
import os
import requests
from pathlib import Path

%cd /kaggle/working/Routing101

# 1. Tải và cài đặt cloudflared tunnel client (nếu chưa có)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || true

# 2. Khởi động FastAPI Backend với Uvicorn
print("🚀 Đang khởi động FastAPI Backend (Routing101)...")
print("   ⏳ Trên CPU, quá trình nạp SigLIP2 + FAISS indexes có thể mất 3-5 phút. Hãy kiên nhẫn!")
backend_log_path = "/kaggle/working/backend.log"
backend_log = open(backend_log_path, "w")
backend_proc = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/Routing101",
    env=os.environ.copy()
)

# 3. Khởi chạy Cloudflare Tunnel để expose port 8000 ra Internet
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_t = time.time()
while time.time() - start_t < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

# 4. HEALTH CHECK: Đợi đến khi Uvicorn hoàn tất nạp model (timeout 300s = 5 phút)
print("⏳ Đang đợi Backend tải xong dữ liệu và mở cổng 8000...")
backend_ready = False
last_log_pos = 0
for attempt in range(300):
    if backend_proc.poll() is not None:
        print("\n❌ Backend đã dừng đột ngột! Xem log bên dưới:")
        print("-" * 60)
        print(open(backend_log_path).read()[-3000:])
        print("-" * 60)
        break
    try:
        resp = requests.get("http://127.0.0.1:8000/docs", timeout=1)
        if resp.status_code == 200:
            backend_ready = True
            break
    except Exception:
        pass
    # Hiển thị tiến trình từ backend.log
    if attempt > 0 and attempt % 5 == 0:
        try:
            with open(backend_log_path, "r") as lf:
                lf.seek(last_log_pos)
                new_lines = lf.read()
                last_log_pos = lf.tell()
                for l in new_lines.strip().split("\n"):
                    if "[startup]" in l:
                        print(f"   📌 {l.strip()}")
        except Exception:
            pass
    if attempt > 0 and attempt % 30 == 0:
        elapsed = int(time.time() - start_t)
        print(f"   ... đã đợi {elapsed}s, backend vẫn đang nạp dữ liệu...")
    time.sleep(1)

# 5. Hiển thị link truy cập
elapsed_total = int(time.time() - start_t)
if backend_ready and public_url:
    print("\n" + "=" * 70)
    print(f"🎉 WEB APP ROUTING101 ĐÃ KHỞI ĐỘNG XONG SAU {elapsed_total}s!")
    print("=" * 70)
    print(f"🔗 TRUY CẬP GIAO DIỆN TẠI:  {public_url}/app/")
    print(f"📑 API SWAGGER DOCS TẠI  :  {public_url}/docs")
    print("=" * 70 + "\n")
elif not public_url:
    print("❌ Không tạo được Cloudflare Tunnel. Hãy kiểm tra kết nối Internet trong cài đặt Notebook!")
else:
    print(f"⚠️ Backend chưa sẵn sàng sau {elapsed_total}s. Xem log cuối cùng:")
    print("-" * 60)
    print(open(backend_log_path).read()[-2000:])
    print("-" * 60)
    print("💡 Nếu log cho thấy vẫn đang load, hãy chạy Bước 6 để theo dõi tiếp!")

## 📜 BƯỚC 6: Xem Logs Trực Tiếp (Real-time Log Viewer)

Chạy cell dưới đây để theo dõi các truy vấn tìm kiếm, RRF fusion, và thời gian thực thi của backend theo thời gian thực:

In [ ]:
# Xem 50 dòng log gần nhất của backend
!tail -n 50 /kaggle/working/backend.log

# Vòng lặp stream log trực tiếp (Nhấn nút Stop/Interrupt trên thanh công cụ để dừng xem log)
try:
    with open("/kaggle/working/backend.log", "r") as f:
        f.seek(0, 2)  # Seek to end
        print("--- BẮT ĐẦU THEO DÕI LOGS (Nhấn Interrupt để thoát) ---")
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\nĐã dừng theo dõi log. Backend vẫn tiếp tục chạy ngầm.")